# Marjum 2026-07 Incremental Scale-Up

Aaron Parsons

Phase 6 of the marjum-2026-07 antenna-position pipeline: batch-fit poses
for (nearly) all photographed images with GPS, then incrementally grow the
joint MCMC fit and watch the antenna position estimate converge.

**Scope note:** a full one-shot joint MCMC over all ~29 images turned out
not to be practical in one sitting -- `horizon_ray_logL` has a per-image,
per-evaluation floor of about 450 ms independent of `n_rays` (dominated by
fixed overhead, not sample count), so an N-image joint fit costs roughly
`N x 0.45s x (draws+tune) x chains`. For a handful of images (Phases 4-5)
that's minutes; for a good fraction of 29 images with enough draws for
real convergence, that's many hours. Speeding up the per-image floor
itself (e.g. batching the ray-tracing across images, as explored and
shelved during Phase 2) is future work. What follows is a genuine
incremental **bootstrap**: three real joint-MCMC runs of growing size,
each with modest draws/chains (a deliberate trade against full
convergence, same caveat as Phases 4-5), showing the estimate actually
improve as images are added -- not a simulation of what scaling up would
do.

In [ ]:
import numpy as np
import matplotlib.pylab as plt
import eigsep_terrain as et
from eigsep_terrain.marjum_dem import MarjumDEM as DEM
import arviz

%matplotlib widget
dem = DEM(cache_file='marjum_dem.npz')
meta = et.meta.load_meta('meta.json')
SITE1P = np.array([1652.80, 2024.20, 1781.60])  # known platform, from EIGSEP Horizons.ipynb

## Batch pose-fitting: 29 of 35 images

Every marjum-2026-07 image with a GPS fix (29 of 35 -- `2209`/`2210` have
none, and `2229`/`2233`/`2236`/`2244` are referenced in the Phase 1 EXIF
cache but no longer present on disk) was run through Phase 2's
`HorizonImage.refine_pose`, cached per-image exactly as in Phase 2. Most
converge well; a substantial minority land on badly-fit poses (SSD in the
hundreds to hundreds of thousands, vs. ~10-90 for a good fit) -- almost
certainly the same position/orientation degeneracy discussed in Phases 2,
4, and 5.1, landing on a worse point along that degenerate ridge for some
images than others, though a few of the worst outliers may have other
issues (unusual lighting, segmentation failures) not yet individually
diagnosed. Rather than chase down each one now, this phase uses only the
images with an SSD low enough to trust (roughly <100), and treats the rest
as a known limitation -- exactly the kind of problem tie points and GPS-
delta priors (Phase 5.1) are suited to fixing, image by image, as future
work.

In [ ]:
import glob, os

def all_gps_keys():
    d = np.load('marjum_2026_07_exif.npz', allow_pickle=True)
    present = set(os.path.splitext(os.path.basename(f))[0].split('_')[-1]
                  for f in glob.glob('marjum-2026-07/*.HEIC'))
    return [k for k, hg in zip(d['keys'], d['has_gps']) if hg and k in present]

ssds = {}
for key in all_gps_keys():
    cache = f'pose_IMG_{key}.npz'
    if os.path.exists(cache):
        ssds[key] = float(np.load(cache)['ssd'])

good_keys = sorted([k for k, s in ssds.items() if s < 100], key=lambda k: ssds[k])
bad_keys = sorted([k for k, s in ssds.items() if s >= 100], key=lambda k: -ssds[k])
print(f'{len(ssds)} images fit, {len(good_keys)} good (SSD<100), {len(bad_keys)} poorly-fit')
print('good:', [(k, round(ssds[k], 1)) for k in good_keys])
print('poorly-fit:', [(k, round(ssds[k], 1)) for k in bad_keys])

fig, ax = plt.subplots(figsize=(10, 4))
keys_by_ssd = sorted(ssds, key=lambda k: ssds[k])
colors = ['green' if ssds[k] < 100 else 'red' for k in keys_by_ssd]
ax.bar(range(len(keys_by_ssd)), [ssds[k] for k in keys_by_ssd], color=colors)
ax.set_xticks(range(len(keys_by_ssd)))
ax.set_xticklabels(keys_by_ssd, rotation=90, fontsize=7)
ax.set_yscale('log')
ax.set_ylabel('horizon fit SSD (log scale)')
ax.axhline(100, color='k', ls=':', lw=1)
ax.set_title('Phase 2 pose-fit quality across all attempted images\n(green=used going forward, red=excluded)')
plt.tight_layout()

## Incremental bootstrap: three real joint-MCMC runs

Each step re-runs the full joint fit (`PositionSolver`: horizon + antenna
+ tie-point + GPS-delta likelihoods together) from scratch with more
images, not a continuation of the previous run's chain. Tie points connect
`2223`<->`2224` (Phase 5.1's original pair) and `2237`<->`2238` (a second,
much less severely-degenerate pair found the same way); GPS-delta priors
chain every other temporally-adjacent pair in the growing set. Traces are
cached (`phase6_step{1,2,4}_trace.nc`) since each takes 15-90 minutes.

In [ ]:
steps = [
    ('phase6_step1_trace.nc', 3, ['2223', '2224', '2238']),
    ('phase6_step2_trace.nc', 6, ['2220', '2223', '2224', '2237', '2238', '2239']),
    ('phase6_step4_trace.nc', 17, ['2213', '2215', '2216', '2217', '2218', '2220', '2221',
                                    '2222', '2223', '2224', '2226', '2231', '2234', '2235',
                                    '2237', '2238', '2239']),
]

results = []
for cache, n_imgs, keys in steps:
    trc = arviz.from_netcdf(cache)
    ant_e = np.asarray(trc.posterior['ant_e']).flatten()
    ant_n = np.asarray(trc.posterior['ant_n']).flatten()
    dist = np.hypot(ant_e.mean() - SITE1P[0], ant_n.mean() - SITE1P[1])
    results.append(dict(n_imgs=n_imgs, ant_e=ant_e, ant_n=ant_n,
                         mean=(ant_e.mean(), ant_n.mean()),
                         std=(ant_e.std(), ant_n.std()), dist=dist))
    print(f'{n_imgs:2d} images: ant=({ant_e.mean():.2f}+/-{ant_e.std():.2f}, '
          f'{ant_n.mean():.2f}+/-{ant_n.std():.2f})  dist from known platform={dist:.2f} m')

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 4.5))
n_imgs_list = [r['n_imgs'] for r in results]
axes[0].plot(n_imgs_list, [r['dist'] for r in results], 'o-', color='C0')
axes[0].set_xlabel('number of images in joint fit')
axes[0].set_ylabel('distance from known platform [m]')
axes[0].set_title('Point estimate converging')

axes[1].plot(n_imgs_list, [r['std'][0] for r in results], 'o-', label='std(ant_e)')
axes[1].plot(n_imgs_list, [r['std'][1] for r in results], 's-', label='std(ant_n)')
axes[1].set_xlabel('number of images in joint fit')
axes[1].set_ylabel('posterior std [m]')
axes[1].set_title('Posterior uncertainty shrinking')
axes[1].legend()
plt.tight_layout()

## Final (17-image) posterior on the terrain map

In [ ]:
final = results[-1]
final_keys = steps[-1][2]
trace_map = arviz.from_netcdf(steps[-1][0])
cam_colors = plt.cm.tab20(np.linspace(0, 1, len(final_keys)))

fig, ax = plt.subplots(figsize=(10, 9))
e0, n0 = final['mean']
rng = 200
et.plot.terrain_plot(dem, ax=ax, colorbar=True, erng_m=(e0 - rng, e0 + rng), nrng_m=(n0 - rng, n0 + rng))

rng_state = np.random.default_rng(0)
arrow_len = 15  # m, length of each orientation-sample line
for key, color in zip(final_keys, cam_colors):
    e = np.asarray(trace_map.posterior[f'{key}_e']).flatten()
    n = np.asarray(trace_map.posterior[f'{key}_n']).flatten()
    ph = np.asarray(trace_map.posterior[f'{key}_ph']).flatten()
    ax.plot(e, n, '.', color=color, alpha=0.15, ms=3)
    idx = rng_state.choice(len(e), size=min(20, len(e)), replace=False)
    for i in idx:
        de, dn = arrow_len * np.cos(ph[i]), arrow_len * np.sin(ph[i])
        ax.plot([e[i], e[i] + de], [n[i], n[i] + dn], '-', color=color, alpha=0.3, lw=0.8)
    ax.annotate(key, (e.mean(), n.mean()), fontsize=6, color=color,
                xytext=(3, 3), textcoords='offset points')

ax.plot(final['ant_e'], final['ant_n'], 'k.', alpha=0.15, ms=4, label='antenna position (posterior)')
ax.plot([], [], '.', color='gray', label='camera position (posterior samples)')
ax.plot([], [], '-', color='gray', label='camera heading (posterior samples)')
ax.legend(loc='best')
ax.set_title('Phase 6: 17-image joint posterior -- camera positions, headings, and antenna position')
plt.tight_layout()

## Diagnostics: model horizon vs actual, per image (17-image fit)

For every image in the final (17-image) joint fit, set its pose to the
posterior mean from `phase6_step4_trace.nc` and compare the ray-traced
sky/ground boundary against the actual segmentation -- the same check used
throughout Phases 2, 4, and 5.1, applied here to confirm the joint fit
didn't quietly degrade any individual image's horizon match while
optimizing the group.

In [ ]:
from eigsep_terrain.img import HorizonImage, PRM_ORDER

STEP4_KEYS = ['2213', '2215', '2216', '2217', '2218', '2220', '2221', '2222', '2223',
              '2224', '2226', '2231', '2234', '2235', '2237', '2238', '2239']
trace4 = arviz.from_netcdf('phase6_step4_trace.nc')

def posterior_mean_pose(img, key, trace):
    d = {}
    for k in PRM_ORDER:
        if k == 'u':
            e = np.asarray(trace.posterior[f'{key}_e']).flatten().mean()
            n = np.asarray(trace.posterior[f'{key}_n']).flatten().mean()
            logh = np.asarray(trace.posterior[f'{key}_log_h']).flatten().mean()
            u0 = float(dem.interp_alt(np.array([e]), np.array([n]))[0])
            d[k] = np.exp(logh) + u0
        else:
            d[k] = np.asarray(trace.posterior[f'{key}_{k}']).flatten().mean()
    img.set_prms([d[k] for k in PRM_ORDER])

step4_imgs = {}
for key in STEP4_KEYS:
    img = HorizonImage(f'marjum-2026-07/IMG_{key}.HEIC', meta=meta, px_smooth=150, px_dist=30)
    posterior_mean_pose(img, key, trace4)
    step4_imgs[key] = img

In [ ]:
ncols = 5
nrows = int(np.ceil(len(STEP4_KEYS) / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(3.2 * ncols, 2.6 * nrows))
axes = axes.ravel()

for ax, key in zip(axes, STEP4_KEYS):
    img = step4_imgs[key]
    sl = slice(None, None, 6)
    actual_sky = img.sky_mask[sl, sl]
    r = img.ray_distance(dem, img.get_rays()[..., sl, sl])
    model_sky = np.isnan(r)
    ax.imshow(img.img[sl, sl], origin='lower')
    ax.contour(actual_sky.astype(float), levels=[0.5], colors='lime', linewidths=1)
    ax.contour(model_sky.astype(float), levels=[0.5], colors='red', linewidths=1)
    bnd_a = img._raster_boundary(actual_sky)
    bnd_p = img._raster_boundary(model_sky)
    m = ~np.isnan(bnd_a) & ~np.isnan(bnd_p)
    ssd = np.mean((bnd_a[m] - bnd_p[m])**2) if m.sum() else np.nan
    ax.set_title(f'{key} (SSD={ssd:.0f})', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

for ax in axes[len(STEP4_KEYS):]:
    ax.axis('off')

fig.suptitle('green=actual segmentation, red=posterior-mean-pose prediction (17-image joint fit)')
plt.tight_layout()

### Where the antenna was identified

The three images with a Phase 3 antenna pick (`2223`, `2224`, `2238`) --
the only ones contributing an `ant_logL` term to the fit -- zoomed on the
picked pixel (magenta `+`), at each image's posterior-mean pose.

In [ ]:
ant_pick_keys = ['2223', '2224', '2238']
zoom_half = 150  # px

fig, axes = plt.subplots(ncols=len(ant_pick_keys), figsize=(5 * len(ant_pick_keys), 5))
for ax, key in zip(axes, ant_pick_keys):
    img = step4_imgs[key]
    x, y = img.meta['ant_px']
    ax.imshow(img.img, origin='lower')
    ax.plot(x, y, 'm+', ms=20, mew=2)
    ax.set_xlim(x - zoom_half, x + zoom_half)
    ax.set_ylim(y - zoom_half, y + zoom_half)
    ax.set_title(f'IMG_{key}: ant_px = ({x:.0f}, {y:.0f})')
plt.tight_layout()

## Posterior corner plots

Antenna position first, then individual camera pose parameters for the
two tie-pointed pairs (`2223`/`2224` and `2237`/`2238`) -- these are the
images where the position/orientation degeneracy was most directly
diagnosed (Phase 5.1), so their posteriors are the most informative check
on whether the joint fit actually constrains them well or just relocates
the same degenerate ridge.

In [ ]:
import corner

vars_to_plot = ['ant_e', 'ant_n', 'ant_log_h']
posterior = trace4.posterior[vars_to_plot].stack(sample=('chain', 'draw'))
samples = np.column_stack([posterior[v].values for v in vars_to_plot])
corner.corner(samples, labels=vars_to_plot, show_titles=True);
plt.suptitle('Antenna position posterior (17-image joint fit)', y=1.02);

In [ ]:
for key in ['2223', '2224', '2237', '2238']:
    vars_to_plot = [f'{key}_e', f'{key}_n', f'{key}_th', f'{key}_ph']
    posterior = trace4.posterior[vars_to_plot].stack(sample=('chain', 'draw'))
    samples = np.column_stack([posterior[v].values for v in vars_to_plot])
    fig = corner.corner(samples, labels=vars_to_plot, show_titles=True)
    fig.suptitle(f'IMG_{key} pose posterior', y=1.02)
    plt.show()

## Findings

- Batch-fitting Phase 2's `refine_pose` across all 29 GPS-tagged images
  (4 more referenced in the Phase 1 cache no longer exist on disk) took
  roughly 25-40 minutes per image under 4-way parallelism on this
  machine's 12 cores -- and running 4 workers concurrently pushed the
  system into swap-thrashing (confirmed via a kernel OOM-kill log entry
  when a concurrent MCMC attempt pushed memory over the edge), which
  itself slowed the batch down further. A rough guide for next time:
  either fewer concurrent workers, or don't run other memory-heavy jobs
  (segmentation models, MCMC) alongside a wide batch-fit.
- 17 of 29 attempted images converged to a trustworthy pose (SSD < 100);
  the rest landed on clearly bad fits (SSD from ~200 to ~190,000). This
  matches the single-image position/orientation degeneracy identified in
  Phase 2 -- some images' independent fits simply land on a worse point of
  that degenerate ridge than others. Diagnosing/fixing each poorly-fit
  image individually (most likely via the same tie-point/GPS-delta
  approach validated in Phase 5.1) is future work, not attempted here.
- The bootstrap itself is the headline result: growing the joint MCMC from
  3 to 6 to 17 images brought the antenna position estimate from 13.9 m to
  3.2 m to **2.7 m** from the previously-known Site 1 platform location,
  with posterior uncertainty shrinking from several meters to well under
  a meter in each of e and n. This is real evidence the pipeline
  converges toward a consistent answer as more images are added, not an
  assumption -- exactly what Phase 4's and Phase 5.1's convergence
  caveats hoped more data would eventually demonstrate.
- All three steps used deliberately modest MCMC settings (2 chains, a few
  hundred draws) for tractable runtime, and PyMC flags the same Rhat/ESS
  convergence caveats as Phases 4-5 at every step. The trend across steps
  is real, but any single step's reported uncertainty should still be
  read as a lower bound, not a final number.
- **Natural next steps, in rough priority order:** (1) diagnose/refit the
  12 poorly-fit images so all 29 can contribute; (2) add tie-point pairs
  beyond the two found so far -- several of the 17 good images were shot
  within seconds of each other (e.g. the `2216`-`2222` cluster) and likely
  overlap; (3) if a full-precision, all-image final answer is wanted,
  invest in batching the ray-tracing likelihood across images to remove
  the ~450 ms/image/eval floor, then run proper long chains (4+, thousands
  of draws) on the full set.
- **Diagnostics added:** per-image model-vs-actual horizon overlays (all
  17 images, at posterior-mean pose) confirm the joint fit didn't quietly
  break any individual image's horizon match -- `2231` is the one visible
  exception (SSD=279, clearly off in the overlay grid), consistent with it
  already being the weakest of the 17 at fit time (SSD=89.5); the
  antenna-position corner plot shows the joint 3D posterior shape (not
  just marginal means/stds); and per-image pose corner plots for the two
  tie-pointed pairs make the point/orientation degeneracy directly
  visible: `2223`, `2224`, `2237`, and `2238` all show the same diagonal
  e-n-th-ph correlation band seen in Phase 2's single-image analysis --
  tie points and GPS-delta priors have narrowed that band a lot (`2223_e`
  now spans about 0.75 m instead of the ~24 m divergence an independent
  fit produced) but have not eliminated it. That residual band is exactly
  what more images (breaking the degeneracy from additional independent
  directions) should keep shrinking.